In [20]:
from google.colab import drive

# Autoriser l'accès à Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import pandas as pd, io, re
from pathlib import Path

chemin = Path('/content/drive/MyDrive/Salmane Khaoula-Zaaiter Oumaima-Fin3-TP2/FMI DATA.xls')
if not chemin.is_file():
    raise FileNotFoundError(f'Fichier introuvable : {chemin}')

# 1) lire le texte (essayer plusieurs encodages)
texte = None
for enc in ('utf-8-sig', 'utf-16', 'cp1252', 'latin-1'):
    try:
        texte = chemin.read_text(encoding=enc)
        break
    except UnicodeError:
        continue

lignes = [l for l in texte.splitlines() if l.strip()]

# 2) trouver la ligne d'en-tête (elle contient 'Facility' et 'Date of')
i_entete = next(i for i, l in enumerate(lignes) if 'Facility' in l and 'Date of' in l)

# 3) garder seulement les lignes de données (contenant une date au format "Mois jj, aaaa" en deuxième colonne)
# Les lignes de données ont la structure : Nom du programme [TAB] Date de début [TAB] Date de fin [TAB] ...
motif = re.compile(r'^[^\t]+\t\s*[A-Z][a-z]+ \d{1,2}, \d{4}\t')
donnees = [l for l in lignes[i_entete + 2:] if motif.match(l)]

# Recréer le CSV tabulaire à partir de la ligne d'en-tête principale et des données
en_tete_propre = "Facility\tDate of Arrangement\tExpiration Date\tAmount Agreed\tAmount Drawn\tAmount Outstanding"
df = pd.read_csv(io.StringIO('\n'.join([en_tete_propre] + donnees)),
                 sep='\t', thousands=',')

# Formater les colonnes
df.columns = ['programme', 'date_accord', 'date_expiration', 'montant_accorde', 'montant_tire', 'montant_restant']
df['date_accord'] = pd.to_datetime(df['date_accord'].str.strip(), format='%b %d, %Y', errors='coerce')
df['annee_accord'] = df['date_accord'].dt.year

display(df.head(10))
print('Dimensions :', df.shape)

,programme,date_accord,date_expiration,montant_accorde,montant_tire,montant_restant,annee_accord
0,Resilience and Sustainability Facility,2023-09-28,"Apr 02, 2025",1000000,562500,562500,2023
1,Flexible Credit Line,2023-04-03,"Apr 02, 2025",3726200,0,0,2023
2,Precautionary and Liquidity Line,2018-12-17,"Apr 07, 2020",2150800,2150800,537700,2018
3,Precautionary and Liquidity Line,2016-07-22,"Jul 21, 2018",2504000,0,0,2016
4,Precautionary and Liquidity Line,2014-07-28,"Jul 21, 2016",3235100,0,0,2014
5,Precautionary and Liquidity Line,2012-08-03,"Jul 27, 2014",4117400,0,0,2012
6,Standby Arrangement,1992-01-31,"Mar 31, 1993",91980,18396,0,1992
7,Standby Arrangement,1990-07-20,"Mar 31, 1991",100000,48000,0,1990
8,Standby Arrangement,1988-08-30,"Dec 31, 1989",210000,210000,0,1988
9,Standby Arrangement,1986-12-16,"Apr 30, 1988",230000,230000,0,1986


Dimensions : (22, 7)


In [22]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
df.to_sql('avoirs_fmi', conn, index=False, if_exists='replace')

def executer_sql(titre, requete):
    print(f"=== {titre} ===")
    display(pd.read_sql_query(requete, conn))
    print("\n")

# 1. Toutes les observations
executer_sql("1. Toutes les observations", "SELECT * FROM avoirs_fmi;")

# 2. Choix de colonnes + calcul : part du montant tiré dans le montant accordé
executer_sql("2. Part du montant tiré dans le montant accordé", """
SELECT annee_accord, programme, montant_accorde, montant_tire,
       ROUND(montant_tire * 100.0 / montant_accorde, 1) AS part_tire_pct
FROM avoirs_fmi
WHERE montant_accorde > 0;
""")

# 3. LIMIT : les 3 accords les plus récents
executer_sql("3. Les 3 accords les plus récents", """
SELECT annee_accord, programme, montant_accorde FROM avoirs_fmi
ORDER BY annee_accord DESC
LIMIT 3;
""")

# 4. ORDER BY : classement selon le montant accordé
executer_sql("4. Tri selon le montant accordé (décroissant)", """
SELECT annee_accord, programme, montant_accorde
FROM avoirs_fmi
ORDER BY montant_accorde DESC;
""")

# 5. WHERE : accords où le montant accordé est inférieur à 100 millions (100 000 milliers)
executer_sql("5. Accords de moins de 100 millions de DTS", """
SELECT annee_accord, programme, montant_accorde FROM avoirs_fmi
WHERE montant_accorde < 100000
ORDER BY annee_accord;
""")

# 6. COUNT : nombre d'accords dépassant 1 milliard (1 000 000 milliers)
executer_sql("6. Nombre d'accords > 1 milliard de DTS", """
SELECT COUNT(*) AS nb_accords FROM avoirs_fmi
WHERE montant_accorde > 1000000;
""")

# 7. AVG : moyenne des montants accordés depuis 2010
executer_sql("7. Moyenne des montants accordés depuis 2010", """
SELECT ROUND(AVG(montant_accorde)) AS moyenne_accorde
FROM avoirs_fmi
WHERE annee_accord >= 2010;
""")

# 8. MIN / MAX des montants accordés
executer_sql("8. Minimum et maximum des montants accordés", """
SELECT MIN(montant_accorde) AS min_accorde, MAX(montant_accorde) AS max_accorde
FROM avoirs_fmi;
""")

# 9. GROUP BY : moyenne du montant accordé par programme
executer_sql("9. Moyenne du montant accordé par programme", """
SELECT programme,
       COUNT(*) AS nb_accords,
       ROUND(AVG(montant_accorde)) AS moyenne_accordee
FROM avoirs_fmi
GROUP BY programme
ORDER BY moyenne_accordee DESC;
""")

# 10. HAVING : programmes ayant eu au moins un accord individuel supérieur à 1 milliard
executer_sql("10. Programmes avec un accord max ayant dépassé 1 milliard de DTS", """
SELECT programme,
       MAX(montant_accorde) AS max_accorde
FROM avoirs_fmi
GROUP BY programme
HAVING MAX(montant_accorde) > 1000000
ORDER BY max_accorde DESC;
""")

# 11. AND / OR
executer_sql("11. AND : depuis 2010 et montant accordé inférieur à 2.5 milliards", """
SELECT annee_accord, programme, montant_accorde FROM avoirs_fmi
WHERE annee_accord >= 2010 AND montant_accorde < 2500000
ORDER BY annee_accord;
""")

executer_sql("11. OR : montant accordé > 2 milliards OU montant restant > 500 millions", """
SELECT annee_accord, programme, montant_accorde, montant_restant FROM avoirs_fmi
WHERE montant_accorde > 2000000 OR montant_restant > 500000
ORDER BY annee_accord;
""")

# 12. BETWEEN et IN
executer_sql("12. BETWEEN : période d'accord 2015-2019", """
SELECT annee_accord, programme, montant_accorde FROM avoirs_fmi
WHERE annee_accord BETWEEN 2015 AND 2019
ORDER BY annee_accord;
""")

executer_sql("12. IN : années choisies", """
SELECT annee_accord, programme, montant_accorde FROM avoirs_fmi
WHERE annee_accord IN (1990, 2012, 2018, 2023)
ORDER BY annee_accord;
""")

# 13. RANK : classement des accords selon le montant accordé
executer_sql("13. Classement des 8 plus gros accords accordés", """
SELECT annee_accord, programme, montant_accorde,
       RANK() OVER (ORDER BY montant_accorde DESC) AS rang
FROM avoirs_fmi
ORDER BY rang
LIMIT 8;
""")

# 14. Sous-requête : accords sous la moyenne globale
executer_sql("14. Accords dont le montant est sous la moyenne globale", """
SELECT annee_accord, programme, montant_accorde
FROM avoirs_fmi
WHERE montant_accorde < (SELECT AVG(montant_accorde) FROM avoirs_fmi)
ORDER BY annee_accord DESC;
""")

# 15. Synthèse : tendance de l'évolution du montant accordé d'un accord au suivant
executer_sql("15. Synthèse : variation du montant par rapport à l'accord précédent", """
SELECT annee_accord, programme, montant_accorde, variation,
  CASE WHEN variation IS NULL THEN 'Premier accord'
       WHEN variation > 0 THEN 'En hausse'
       WHEN variation < 0 THEN 'En baisse'
       ELSE 'Identique' END AS tendance
FROM (
    SELECT annee_accord, programme, montant_accorde,
           montant_accorde - LAG(montant_accorde) OVER (ORDER BY annee_accord) AS variation
    FROM avoirs_fmi
)
ORDER BY annee_accord DESC;
""")

=== 1. Toutes les observations ===


,programme,date_accord,date_expiration,montant_accorde,montant_tire,montant_restant,annee_accord
0,Resilience and Sustainability Facility,2023-09-28 00:00:00,"Apr 02, 2025",1000000,562500,562500,2023
1,Flexible Credit Line,2023-04-03 00:00:00,"Apr 02, 2025",3726200,0,0,2023
2,Precautionary and Liquidity Line,2018-12-17 00:00:00,"Apr 07, 2020",2150800,2150800,537700,2018
3,Precautionary and Liquidity Line,2016-07-22 00:00:00,"Jul 21, 2018",2504000,0,0,2016
4,Precautionary and Liquidity Line,2014-07-28 00:00:00,"Jul 21, 2016",3235100,0,0,2014
5,Precautionary and Liquidity Line,2012-08-03 00:00:00,"Jul 27, 2014",4117400,0,0,2012
6,Standby Arrangement,1992-01-31 00:00:00,"Mar 31, 1993",91980,18396,0,1992
7,Standby Arrangement,1990-07-20 00:00:00,"Mar 31, 1991",100000,48000,0,1990
8,Standby Arrangement,1988-08-30 00:00:00,"Dec 31, 1989",210000,210000,0,1988
9,Standby Arrangement,1986-12-16 00:00:00,"Apr 30, 1988",230000,230000,0,1986




=== 2. Part du montant tiré dans le montant accordé ===


,annee_accord,programme,montant_accorde,montant_tire,part_tire_pct
0,2023,Resilience and Sustainability Facility,1000000,562500,56.3
1,2023,Flexible Credit Line,3726200,0,0.0
2,2018,Precautionary and Liquidity Line,2150800,2150800,100.0
3,2016,Precautionary and Liquidity Line,2504000,0,0.0
4,2014,Precautionary and Liquidity Line,3235100,0,0.0
5,2012,Precautionary and Liquidity Line,4117400,0,0.0
6,1992,Standby Arrangement,91980,18396,20.0
7,1990,Standby Arrangement,100000,48000,48.0
8,1988,Standby Arrangement,210000,210000,100.0
9,1986,Standby Arrangement,230000,230000,100.0




=== 3. Les 3 accords les plus récents ===


...

,annee_accord,programme,montant_accorde,rang
0,2012,Precautionary and Liquidity Line,4117400,1
1,2023,Flexible Credit Line,3726200,2
2,2014,Precautionary and Liquidity Line,3235100,3
3,2016,Precautionary and Liquidity Line,2504000,4
4,2018,Precautionary and Liquidity Line,2150800,5
5,2023,Resilience and Sustainability Facility,1000000,6
6,1981,Extended Fund Facility,817050,7
7,1980,Extended Fund Facility,810000,8




=== 14. Accords dont le montant est sous la moyenne globale ===


,annee_accord,programme,montant_accorde
0,1992,Standby Arrangement,91980
1,1990,Standby Arrangement,100000
2,1988,Standby Arrangement,210000
3,1986,Standby Arrangement,230000
4,1985,Standby Arrangement,200000
5,1983,Standby Arrangement,300000
6,1982,Standby Arrangement,281250
7,1981,Extended Fund Facility,817050
8,1980,Extended Fund Facility,810000
9,1971,Standby Arrangement,30000




=== 15. Synthèse : variation du montant par rapport à l'accord précédent ===


,annee_accord,programme,montant_accorde,variation,tendance
0,2023,Resilience and Sustainability Facility,1000000,-1150800.0,En baisse
1,2023,Flexible Credit Line,3726200,2726200.0,En hausse
2,2018,Precautionary and Liquidity Line,2150800,-353200.0,En baisse
3,2016,Precautionary and Liquidity Line,2504000,-731100.0,En baisse
4,2014,Precautionary and Liquidity Line,3235100,-882300.0,En baisse
5,2012,Precautionary and Liquidity Line,4117400,4025420.0,En hausse
6,1992,Standby Arrangement,91980,-8020.0,En baisse
7,1990,Standby Arrangement,100000,-110000.0,En baisse
8,1988,Standby Arrangement,210000,-20000.0,En baisse
9,1986,Standby Arrangement,230000,30000.0,En hausse
